# Baselines clásicos — Random Forest, Linear SVM y OBIA

Notebook todo-en-uno para entrenar y evaluar los tres baselines tradicionales sobre el **mismo split de train/val/test** que los modelos de Deep Learning (U-Net, DeepLabv3+, U-Net++).

**Objetivo:** producir una comparativa rigurosa contra los modelos profundos sin depender de valores reportados en literatura externa.

## Contenido

1. Setup y verificación del entorno
2. Resumen del dataset (parches por split y por clase)
3. **Random Forest** (píxel-a-píxel, 15 features handcrafted)
4. **Linear SVM** via SGDClassifier (mismas features)
5. **OBIA** (SLIC superpíxeles + RF sobre segmentos)
6. Comparativa entre los tres baselines + DL

**Tiempo total estimado:** 6–10 h en CPU multinúcleo (ajustable bajando `N_PATCHES_TRAIN` y `MAX_TEST_PATCHES`).

Para una corrida rápida de validación (~10 min) usa la sección **Quick test** al final.

## 1. Setup

In [ ]:
# Configuración global de la corrida
# Ajusta estos valores según el tiempo disponible.

N_PATCHES_TRAIN = 600              # parches de train usados para sampling
N_PER_CLASS_PER_PATCH = 80          # píxeles por clase por parche
MAX_TEST_PATCHES = None             # None = todos los 18991 parches; pon ej. 1000 para rápido

RF_N_ESTIMATORS = 200
RF_MAX_DEPTH = 25

SVM_MAX_ITER = 200

SLIC_N_SEGMENTS = 150
SLIC_COMPACTNESS = 10.0
OBIA_N_PATCHES_TRAIN = 600

SEED = 42

print(f'Configuración:')
print(f'  N_PATCHES_TRAIN={N_PATCHES_TRAIN}, N_PER_CLASS_PER_PATCH={N_PER_CLASS_PER_PATCH}')
print(f'  MAX_TEST_PATCHES={MAX_TEST_PATCHES}')
print(f'  RF: {RF_N_ESTIMATORS} árboles, max_depth={RF_MAX_DEPTH}')
print(f'  SLIC: n_segments={SLIC_N_SEGMENTS}, compactness={SLIC_COMPACTNESS}')
print(f'  Seed: {SEED}')

In [ ]:
# Imports y paths
import sys, os, time, json
from pathlib import Path
from datetime import datetime

ROOT = Path(os.path.abspath('..')).resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'scripts' / '02_training'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import torch

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from skimage.segmentation import slic

from scripts.utils.handcrafted_features import (
    extract_pixel_features, extract_segment_features, majority_label_per_segment,
    PIXEL_FEATURE_NAMES, SEGMENT_FEATURE_NAMES,
)
from scripts.utils import N_CLASSES
from scripts.utils.config import CHECKPOINTS_DIR
from scripts.utils.metrics import SegmentationMetrics
from _baseline_common import (
    CLASS_NAMES_FULL, load_index, load_patch, sample_pixels_stratified,
)

print(f'ROOT: {ROOT}')
print(f'Píxel features: {len(PIXEL_FEATURE_NAMES)} → {PIXEL_FEATURE_NAMES}')
print(f'Segment features: {len(SEGMENT_FEATURE_NAMES)} → {SEGMENT_FEATURE_NAMES}')
print(f'Clases ({N_CLASSES}): {CLASS_NAMES_FULL}')

## 2. Resumen del dataset

In [ ]:
df_train = load_index('train')
df_val = load_index('val')
df_test = load_index('test')

print(f'Train:  {len(df_train):>6,} parches')
print(f'Val:    {len(df_val):>6,} parches')
print(f'Test:   {len(df_test):>6,} parches')
print()
print('Distribución por municipio (test):')
print(df_test["ciudad"].value_counts().to_string())

## 3. Random Forest (píxel-a-píxel)

Entrenamos un Random Forest sobre 15 features handcrafted por píxel:
- 3 RGB crudos
- 6 estadísticas locales (mean, std) en ventana 5×5
- 3 magnitudes de gradiente Sobel
- 3 Laplaciano por canal

Sampling: ~480 K muestras balanceadas por clase (8 clases × 80 px × 600 parches).

In [ ]:
# 3.1 Sampling estratificado para RF/SVM
print(f'Sampling píxeles ({N_PATCHES_TRAIN} parches × {N_PER_CLASS_PER_PATCH} px/clase × 8 clases)...')
t0 = time.time()
X_train, y_train = sample_pixels_stratified(
    df_train,
    feature_fn=extract_pixel_features,
    n_patches=N_PATCHES_TRAIN,
    n_per_class_per_patch=N_PER_CLASS_PER_PATCH,
    rng_seed=SEED,
    verbose=True,
)
print(f'X_train: {X_train.shape}   y_train: {y_train.shape}')
print(f'Distribución de clases (train):')
for cls in range(N_CLASSES):
    n = int((y_train == cls).sum())
    if n > 0:
        print(f'  {CLASS_NAMES_FULL[cls]:20s}  {n:>8,}')
print(f'Tiempo: {time.time()-t0:.1f}s')

In [ ]:
# 3.2 Entrenamiento RF
print(f'Entrenando RandomForestClassifier (n_estimators={RF_N_ESTIMATORS}, max_depth={RF_MAX_DEPTH})...')
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    n_jobs=-1,
    random_state=SEED,
    verbose=0,
)
rf.fit(X_train, y_train)
print(f'Train accuracy: {rf.score(X_train, y_train):.4f}')
print(f'Tiempo entrenamiento: {time.time()-t0:.1f}s')

out_dir = Path(CHECKPOINTS_DIR) / 'rf_baseline'
out_dir.mkdir(parents=True, exist_ok=True)
joblib.dump({
    'model': rf, 'feature_extractor': 'extract_pixel_features',
    'n_features': X_train.shape[1], 'class_names': CLASS_NAMES_FULL, 'n_classes': N_CLASSES,
    'config': {'n_patches': N_PATCHES_TRAIN, 'n_per_class_per_patch': N_PER_CLASS_PER_PATCH,
               'n_estimators': RF_N_ESTIMATORS, 'max_depth': RF_MAX_DEPTH, 'seed': SEED},
}, out_dir / 'rf.joblib', compress=3)
print(f'Modelo guardado: {out_dir / "rf.joblib"}')

In [ ]:
# 3.3 Importancia de features (top-10)
importances = pd.Series(rf.feature_importances_, index=PIXEL_FEATURE_NAMES).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color='#2E86C1')
ax.set_xlabel('Importancia')
ax.set_title('Random Forest — importancia de features')
ax.invert_yaxis()
plt.tight_layout()
plt.show()
print(importances.to_string())

In [ ]:
# 3.4 Evaluación RF sobre test set
def evaluate_pixelwise(model, df_test, max_patches=None, model_name='model'):
    if max_patches:
        df = df_test.iloc[:max_patches]
    else:
        df = df_test
    print(f'Evaluando {model_name} sobre {len(df):,} parches de test...')
    metrics = SegmentationMetrics(
        num_classes=N_CLASSES, ignore_index=0,
        class_names={i: CLASS_NAMES_FULL[i] for i in range(N_CLASSES)},
    )
    t0 = time.time()
    skipped = 0
    for i, row in enumerate(df.itertuples(index=False), start=1):
        try:
            img, mask = load_patch(row.image_path, row.mask_path)
        except FileNotFoundError:
            skipped += 1; continue
        feats = extract_pixel_features(img)
        pred_flat = model.predict(feats).astype(np.uint8)
        pred = pred_flat.reshape(img.shape[:2])
        metrics.update(
            torch.from_numpy(pred[None]).long(),
            torch.from_numpy(mask[None]).long(),
        )
        if i % 500 == 0:
            elapsed = time.time() - t0
            eta = elapsed / i * (len(df) - i)
            print(f'  [{i:>5d}/{len(df)}]  elapsed={elapsed/60:.1f}min  ETA={eta/60:.1f}min')
    print(f'  Tiempo total: {(time.time()-t0)/60:.1f} min  (skipped: {skipped})')
    return metrics

metrics_rf = evaluate_pixelwise(rf, df_test, max_patches=MAX_TEST_PATCHES, model_name='Random Forest')
metrics_rf.print_summary()

In [ ]:
# 3.5 Guardar resultados RF
def save_metrics(metrics, model_name, n_test, extra=None):
    iou = metrics.get_iou_per_class()
    dice = metrics.get_dice_per_class()
    precision = metrics.get_precision_per_class()
    recall = metrics.get_class_accuracy()
    f1 = metrics.get_f1_per_class()
    per_class = {}
    for c in range(N_CLASSES):
        per_class[CLASS_NAMES_FULL[c]] = {
            'IoU': float(iou[c]) if not np.isnan(iou[c]) else 0.0,
            'Dice': float(dice[c]) if not np.isnan(dice[c]) else 0.0,
            'Precision': float(precision[c]) if not np.isnan(precision[c]) else 0.0,
            'Recall': float(recall[c]) if not np.isnan(recall[c]) else 0.0,
            'F1': float(f1[c]) if not np.isnan(f1[c]) else 0.0,
        }
    payload = {
        'model_name': model_name,
        'test_date': datetime.utcnow().isoformat(),
        'test_patches': n_test,
        'metrics': {
            'mIoU': float(metrics.get_miou(ignore_background=True)),
            'mIoU_with_bg': float(metrics.get_miou(ignore_background=False)),
            'pixel_accuracy': float(metrics.get_pixel_accuracy()),
        },
        'per_class': per_class,
    }
    if extra: payload.update(extra)
    out_dir = ROOT / 'results' / 'test_evaluation' / model_name
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / 'confusion_matrix.npy', metrics.confusion_matrix)
    with open(out_dir / 'test_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    print(f'  → {out_dir}/test_metrics.json')
    return payload

n_test_evaluated = MAX_TEST_PATCHES if MAX_TEST_PATCHES else len(df_test)
result_rf = save_metrics(
    metrics_rf, 'rf_baseline', n_test_evaluated,
    extra={'feature_extractor': 'extract_pixel_features',
           'config': {'n_estimators': RF_N_ESTIMATORS, 'max_depth': RF_MAX_DEPTH}},
)

## 4. Linear SVM (SGDClassifier hinge loss)

El SVM con kernel RBF no escala a cientos de miles de píxeles. Usamos `SGDClassifier(loss='hinge')` que aproxima un Linear SVM y soporta millones de muestras.

Mismas 15 features que RF, con `StandardScaler` y `class_weight='balanced'`.

In [ ]:
# 4.1 Entrenamiento SVM
print(f'Entrenando Linear SVM (SGD-hinge, max_iter={SVM_MAX_ITER})...')
t0 = time.time()
svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SGDClassifier(
        loss='hinge', alpha=1e-4, max_iter=SVM_MAX_ITER, tol=1e-4,
        n_jobs=-1, random_state=SEED, class_weight='balanced', verbose=0,
    )),
])
svm.fit(X_train, y_train)
print(f'Train accuracy: {svm.score(X_train, y_train):.4f}')
print(f'Tiempo entrenamiento: {time.time()-t0:.1f}s')

out_dir = Path(CHECKPOINTS_DIR) / 'svm_baseline'
out_dir.mkdir(parents=True, exist_ok=True)
joblib.dump({
    'model': svm, 'feature_extractor': 'extract_pixel_features',
    'n_features': X_train.shape[1], 'class_names': CLASS_NAMES_FULL, 'n_classes': N_CLASSES,
    'config': {'max_iter': SVM_MAX_ITER, 'seed': SEED, 'model_type': 'SGDClassifier(hinge)'},
}, out_dir / 'svm.joblib', compress=3)
print(f'Modelo guardado: {out_dir / "svm.joblib"}')

In [ ]:
# 4.2 Evaluación SVM
metrics_svm = evaluate_pixelwise(svm, df_test, max_patches=MAX_TEST_PATCHES, model_name='Linear SVM')
metrics_svm.print_summary()

In [ ]:
# 4.3 Guardar resultados SVM
result_svm = save_metrics(
    metrics_svm, 'svm_baseline', n_test_evaluated,
    extra={'feature_extractor': 'extract_pixel_features',
           'config': {'max_iter': SVM_MAX_ITER, 'model_type': 'SGDClassifier(hinge)'}},
)

## 5. OBIA (Object-Based Image Analysis)

Pipeline:
1. Por cada parche: SLIC superpixels (~150 segmentos/parche).
2. Por cada segmento: 14 features (mean/std RGB, gradiente, área, perímetro, compactness, posición).
3. Label del segmento = clase mayoritaria de píxeles dentro.
4. RandomForest sobre los vectores de segmento.

In [ ]:
# 5.1 Generar dataset OBIA (segmentos + features) sobre train
print(f'OBIA: SLIC + features sobre {OBIA_N_PATCHES_TRAIN} parches train...')
df_obia = df_train.sample(frac=1.0, random_state=SEED).iloc[:OBIA_N_PATCHES_TRAIN]
t0 = time.time()
X_obia, y_obia = [], []
skipped = 0
for i, row in enumerate(df_obia.itertuples(index=False), start=1):
    try:
        img, mask = load_patch(row.image_path, row.mask_path)
    except FileNotFoundError:
        skipped += 1; continue
    lo, hi = np.percentile(img, (1, 99))
    img_for_slic = np.clip((img - lo) / max(hi-lo, 1e-6), 0, 1)
    segments = slic(img_for_slic, n_segments=SLIC_N_SEGMENTS,
                    compactness=SLIC_COMPACTNESS, start_label=0, channel_axis=-1)
    X, seg_ids = extract_segment_features(img, segments)
    if X.size == 0: continue
    labels_all = majority_label_per_segment(mask, segments, n_classes=N_CLASSES)
    y = labels_all[seg_ids]
    keep = y != 0
    if not keep.any(): continue
    X_obia.append(X[keep]); y_obia.append(y[keep])
    if i % 100 == 0:
        print(f'  [{i}/{len(df_obia)}]  elapsed={(time.time()-t0)/60:.1f}min')
X_obia = np.vstack(X_obia).astype(np.float32)
y_obia = np.concatenate(y_obia).astype(np.int64)
print(f'\nX_obia: {X_obia.shape}   y_obia: {y_obia.shape}')
print(f'Distribución de clases (segmentos):')
for cls in range(N_CLASSES):
    n = int((y_obia == cls).sum())
    if n > 0:
        print(f'  {CLASS_NAMES_FULL[cls]:20s}  {n:>8,}')
print(f'Tiempo: {(time.time()-t0)/60:.1f} min')

In [ ]:
# 5.2 Entrenar RF sobre vectores de segmento
print(f'Entrenando RF sobre segmentos OBIA...')
t0 = time.time()
obia = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS, max_depth=RF_MAX_DEPTH, n_jobs=-1,
    random_state=SEED, class_weight='balanced', verbose=0,
)
obia.fit(X_obia, y_obia)
print(f'Train accuracy (segmentos): {obia.score(X_obia, y_obia):.4f}')
print(f'Tiempo entrenamiento: {time.time()-t0:.1f}s')

out_dir = Path(CHECKPOINTS_DIR) / 'obia_baseline'
out_dir.mkdir(parents=True, exist_ok=True)
joblib.dump({
    'model': obia, 'feature_extractor': 'extract_segment_features',
    'n_features': X_obia.shape[1], 'class_names': CLASS_NAMES_FULL, 'n_classes': N_CLASSES,
    'slic_params': {'n_segments': SLIC_N_SEGMENTS, 'compactness': SLIC_COMPACTNESS, 'start_label': 0},
    'config': {'n_patches': OBIA_N_PATCHES_TRAIN, 'n_estimators': RF_N_ESTIMATORS,
               'max_depth': RF_MAX_DEPTH, 'seed': SEED},
}, out_dir / 'obia.joblib', compress=3)
print(f'Modelo guardado: {out_dir / "obia.joblib"}')

In [ ]:
# 5.3 Evaluación OBIA sobre test
def evaluate_obia(model, slic_params, df_test, max_patches=None):
    df = df_test.iloc[:max_patches] if max_patches else df_test
    print(f'Evaluando OBIA sobre {len(df):,} parches de test...')
    metrics = SegmentationMetrics(
        num_classes=N_CLASSES, ignore_index=0,
        class_names={i: CLASS_NAMES_FULL[i] for i in range(N_CLASSES)},
    )
    t0 = time.time()
    skipped = 0
    for i, row in enumerate(df.itertuples(index=False), start=1):
        try:
            img, mask = load_patch(row.image_path, row.mask_path)
        except FileNotFoundError:
            skipped += 1; continue
        lo, hi = np.percentile(img, (1, 99))
        img_for_slic = np.clip((img - lo) / max(hi-lo, 1e-6), 0, 1)
        segments = slic(img_for_slic, n_segments=slic_params['n_segments'],
                        compactness=slic_params['compactness'],
                        start_label=slic_params.get('start_label', 0), channel_axis=-1)
        X, seg_ids = extract_segment_features(img, segments)
        h, w = img.shape[:2]
        pred_mask = np.zeros((h, w), dtype=np.uint8)
        if X.size > 0:
            seg_preds = model.predict(X).astype(np.uint8)
            seg_to_class = np.zeros(int(segments.max()) + 1, dtype=np.uint8)
            for seg_id, cls in zip(seg_ids, seg_preds):
                seg_to_class[seg_id] = cls
            pred_mask = seg_to_class[segments]
        metrics.update(
            torch.from_numpy(pred_mask[None]).long(),
            torch.from_numpy(mask[None]).long(),
        )
        if i % 500 == 0:
            elapsed = time.time() - t0
            eta = elapsed / i * (len(df) - i)
            print(f'  [{i:>5d}/{len(df)}]  elapsed={elapsed/60:.1f}min  ETA={eta/60:.1f}min')
    print(f'  Tiempo total: {(time.time()-t0)/60:.1f} min')
    return metrics

metrics_obia = evaluate_obia(
    obia, {'n_segments': SLIC_N_SEGMENTS, 'compactness': SLIC_COMPACTNESS, 'start_label': 0},
    df_test, max_patches=MAX_TEST_PATCHES,
)
metrics_obia.print_summary()

In [ ]:
# 5.4 Guardar resultados OBIA
result_obia = save_metrics(
    metrics_obia, 'obia_baseline', n_test_evaluated,
    extra={'feature_extractor': 'extract_segment_features',
           'slic_params': {'n_segments': SLIC_N_SEGMENTS, 'compactness': SLIC_COMPACTNESS, 'start_label': 0},
           'config': {'n_patches': OBIA_N_PATCHES_TRAIN, 'n_estimators': RF_N_ESTIMATORS,
                      'max_depth': RF_MAX_DEPTH}},
)

## 6. Comparativa final — DL vs Baselines clásicos

Junta los resultados de los 3 baselines clásicos con los 3 modelos de DL ya evaluados.

In [ ]:
# 6.1 Cargar las 6 métricas
models_to_compare = [
    ('U-Net (ResNet-34)',         'unet_resnet34_baseline'),
    ('DeepLabv3+ (ResNet-50)',    'deeplabv3_resnet50_baseline'),
    ('U-Net++ (ResNet-50)',       'unetplusplus_resnet50_baseline'),
    ('Random Forest',             'rf_baseline'),
    ('Linear SVM',                'svm_baseline'),
    ('OBIA (SLIC + RF)',          'obia_baseline'),
]

rows = []
for label, mname in models_to_compare:
    p = ROOT / 'results' / 'test_evaluation' / mname / 'test_metrics.json'
    if not p.exists():
        print(f'  [skip] {mname} — sin métricas')
        continue
    with open(p) as f: d = json.load(f)
    row = {'modelo': label}
    row['mIoU'] = d['metrics']['mIoU']
    row['pixel_acc'] = d['metrics']['pixel_accuracy']
    for c in CLASS_NAMES_FULL[1:]:
        row[c] = d['per_class'][c]['IoU']
    rows.append(row)

df_compare = pd.DataFrame(rows).set_index('modelo')
print('Tabla comparativa — IoU por clase + mIoU + pixel accuracy:')
df_compare.round(3)

In [ ]:
# 6.2 Guardar tabla comparativa
out_csv = ROOT / 'results' / 'comparativa_tres_modelos' / 'tabla_dl_vs_baselines.csv'
out_csv.parent.mkdir(parents=True, exist_ok=True)
df_compare.to_csv(out_csv)
print(f'CSV guardado: {out_csv}')

In [ ]:
# 6.3 Gráfico comparativo de IoU por clase
iou_cols = CLASS_NAMES_FULL[1:]
fig, ax = plt.subplots(figsize=(14, 6))
df_compare[iou_cols].T.plot(kind='bar', ax=ax, width=0.85)
ax.set_ylabel('IoU')
ax.set_title('IoU por clase — DL vs baselines clásicos (test set)', fontweight='bold')
ax.set_ylim(0, 1)
ax.legend(loc='upper right', fontsize=9, ncol=2)
ax.grid(True, axis='y', alpha=0.3)
ax.set_xticklabels([c.replace('Cabecera', 'Cabec.').replace('Deposito', 'Dep.') for c in iou_cols],
                    rotation=30, ha='right')
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'comparativa_tres_modelos' / 'fig_dl_vs_baselines.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# 6.4 Resumen final (mIoU)
summary = df_compare[['mIoU', 'pixel_acc']].sort_values('mIoU', ascending=False)
print('\nRanking por mIoU:')
print(summary.round(4).to_string())

fig, ax = plt.subplots(figsize=(10, 4))
summary['mIoU'].plot(kind='barh', ax=ax, color=['#16A34A' if 'Net' in m or 'DeepLab' in m else '#E67E22' for m in summary.index])
ax.set_xlabel('mIoU (8 clases, sin background)')
ax.set_title('mIoU por modelo — DL (verde) vs Baselines clásicos (naranja)')
ax.set_xlim(0, max(summary['mIoU'].max() * 1.1, 0.6))
ax.grid(True, axis='x', alpha=0.3)
for i, v in enumerate(summary['mIoU']):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'comparativa_tres_modelos' / 'fig_miou_ranking.png', dpi=200, bbox_inches='tight')
plt.show()

## ✓ Listo

**Resultados generados:**
- `models/checkpoints/{rf,svm,obia}_baseline/*.joblib` — modelos entrenados
- `results/test_evaluation/{rf,svm,obia}_baseline/test_metrics.json` — métricas
- `results/test_evaluation/{rf,svm,obia}_baseline/confusion_matrix.npy` — matrices
- `results/comparativa_tres_modelos/tabla_dl_vs_baselines.csv` — tabla
- `results/comparativa_tres_modelos/fig_dl_vs_baselines.png` — gráfico por clase
- `results/comparativa_tres_modelos/fig_miou_ranking.png` — ranking mIoU

**Tip:** si el notebook tarda demasiado en evaluación, baja `MAX_TEST_PATCHES` en la celda de configuración (sección 1).